# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access top-level metadata as attributes, not dict keys
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will enumerate all record sets (tables), then within each record set, list the fields (columns) along with their IDs and types, so you can reference them later using their `@id`s.

In [ ]:
# List available record sets (`@id`) and their fields
record_set_metadatas = dataset.record_sets
print(f"Number of record sets: {len(record_set_metadatas)}\n")

for record_set in record_set_metadatas:
    print(f"Record Set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print('-'*40)

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. You'll use each record set's `@id` for referencing and future analysis.

In [ ]:
# Extract data from all record sets and display available columns
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

for record_set_id, df in dataframes.items():
    print(f"\nDataFrame for Record Set: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by key attributes using only `@id` references.

In [ ]:
# Example: Analyze a numeric field from the first available record set.
if record_set_ids:
    analysis_record_set = record_set_ids[0]
    df = dataframes[analysis_record_set]
    print(f"Working with record set {analysis_record_set}")
    
    # Find first numeric field for demonstration
    record_set_obj = next(rs for rs in dataset.record_sets if rs.id == analysis_record_set)
    numeric_fields = [field for field in record_set_obj.fields if field.data_type.lower() in ('float', 'number', 'integer')]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        print(f"Using numeric field: {numeric_field_id}")

        if numeric_field_id in df.columns:
            numeric_field = numeric_field_id

            # Remove rows with missing values for the numeric field
            filtered_df = df[df[numeric_field].notna()].copy()

            # Example threshold: median value
            try:
                # Convert to numeric if necessary
                filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
                threshold = filtered_df[numeric_field].median()
                filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
                print(f"Filtered records with {numeric_field} > {threshold}")
                display(filtered_df.head())

                # Normalize numeric field
                mean = filtered_df[numeric_field].mean()
                std = filtered_df[numeric_field].std()
                filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
                print(f"Normalized {numeric_field} for filtered records:")
                display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

                # Find a categorical (string) field for grouping
                group_fields = [field for field in record_set_obj.fields if field.data_type.lower() == 'text']
                if group_fields:
                    group_field = group_fields[0].id
                    if group_field in filtered_df.columns:
                        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
                        print(f"Grouped data by {group_field} (show first 5 groups):")
                        display(grouped_df.head())
            except Exception as e:
                print(f"Error during numeric field analysis: {e}")
        else:
            print(f"Warning: Numeric field {numeric_field_id} not present in DataFrame columns.")
    else:
        print(f"No numeric fields found in record set {analysis_record_set}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The following is a generic distribution plot of the first numeric field found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_fields:
    # Use the filtered_df and numeric_field from previous cell if available
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {analysis_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()


## 6. Conclusion
Summarize key findings and next steps from the dataset exploration.

- The FAIR^2 dataset provides ordered logistic regression results and related predictors for the adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.
- Using `mlcroissant`, you can systematically explore, query, and analyze all fields and record sets using their `@id`s.
- Further detailed analysis can be performed using the full schema and complete data, including domain-specific grouping, cross-table analyses, or model benchmarking.